# Ground-truth benchmark design

Research question: how does a bundled estimator behave on small synthetic long-range dependence signals with known target values?

This notebook builds a compact manifest in Python, runs it, validates the output contract, and inspects the generated tables.

In [ ]:
from pathlib import Path

import pandas as pd

from lrdbench.output_contract import validate_output_contract
from lrdbench.runner import run_manifest_mapping

In [ ]:
manifest = {
    "manifest_id": "workshop_ground_truth_v1",
    "name": "workshop_ground_truth",
    "mode": "ground_truth",
    "source": {
        "type": "generator_grid",
        "generators": [
            {
                "family": "fGn",
                "params": {"H": [0.5, 0.7], "n": [128], "sigma": [1.0]},
                "replicates": 2,
            }
        ],
    },
    "estimators": [
        {
            "name": "RS",
            "family": "time_domain",
            "target_estimand": "hurst_scaling_proxy",
            "supports_ci": True,
            "params": {"n_bootstrap": 32, "bootstrap_block_len": 12, "ci_levels": [0.95]},
        }
    ],
    "metrics": ["bias", "mae", "rmse", "validity_rate", "runtime", {"name": "ci_width", "levels": [0.95]}],
    "leaderboards": [
        {
            "name": "accuracy_summary",
            "mode": "ground_truth",
            "component_metrics": ["mae", "validity_rate", "runtime"],
            "weights": {"mae": 0.5, "validity_rate": 0.3, "runtime": 0.2},
            "ranking_rule": "weighted_rank",
            "tie_break_rule": "mae",
        }
    ],
    "report": {"formats": ["html", "csv"], "export_root": "reports/notebooks/ground_truth"},
    "seeds": {"global_seed": 20260427},
    "execution": {"max_workers": 1},
}

In [ ]:
out = run_manifest_mapping(manifest, base_dir=Path.cwd())
errors = validate_output_contract(out.result_store_path)
assert errors == []
out.result_store_path

In [ ]:
result_store = Path(out.result_store_path)
metrics = pd.read_csv(result_store / "tables" / "per_stratum_metrics.csv")
leaderboard = pd.read_csv(result_store / "tables" / "leaderboard.csv")
metrics[["estimator_name", "metric_name", "value"]].head(), leaderboard